# **PREPARAÇÃO DE DADOS**

# Considerações iniciais



LOrem

## Tratamento de PARTICIPANTES_2024

In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt


In [ ]:
ano = 2023

In [ ]:
arquivo_microdados = Path('../data/data_raw') / f'MICRODADOS_ENEM_{ano}.csv'
df_microdados = pd.read_csv(arquivo_microdados, sep=';', encoding='latin-1')

In [ ]:
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)

df_microdados.head()

In [ ]:
df_participantes = df_microdados[['IN_TREINEIRO', 'SG_UF_PROVA', 'CO_MUNICIPIO_PROVA', 'Q006', 'NO_MUNICIPIO_PROVA']]
df_resultado = df_microdados[['SG_UF_PROVA', 'NO_MUNICIPIO_PROVA', 'CO_MUNICIPIO_PROVA', 'TP_PRESENCA_CN', 'TP_PRESENCA_CH', 'TP_PRESENCA_LC', 'TP_PRESENCA_MT', 'NU_NOTA_CN', 'NU_NOTA_CH', 'NU_NOTA_LC', 'NU_NOTA_MT', 'NU_NOTA_REDACAO' ]]

df_participantes = df_participantes.rename(columns={'NO_MUNICIPIO_PROVA': 'MUNICIPIO', 'CO_MUNICIPIO_PROVA': 'COD_MUNICIPIO'})
df_resultado = df_resultado.rename(columns={'NO_MUNICIPIO_PROVA': 'MUNICIPIO', 'CO_MUNICIPIO_PROVA': 'COD_MUNICIPIO'})

In [ ]:
df_participantes.info()

In [ ]:
df_participantes.head()

In [ ]:
df_participantes.isnull().mean()*100

In [ ]:
cor_renda_map_sm = {
    "A": 0.0,
    "B": 0.5,
    "C": 1.25,
    "D": 1.75,
    "E": 2.25,
    "F": 2.75,
    "G": 3.5,
    "H": 4.5,
    "I": 5.5,
    "J": 6.5,
    "K": 7.5,
    "L": 8.5,
    "M": 9.5,
    "N": 11.0,
    "O": 13.5,
    "P": 17.5,
    "Q": 20.0 
}
df_participantes['Q006'] = df_participantes['Q006'].map(cor_renda_map_sm)
df_participantes['Q006'] = df_participantes['Q006'].astype("float64")
df_participantes = df_participantes.rename(columns={'Q006': 'RENDA_FAMILIAR_SM'})

In [ ]:
if 'RENDA_FAMILIAR_SM' in df_participantes.columns:
    if not np.issubdtype(df_participantes['RENDA_FAMILIAR_SM'].dropna().dtype, np.number):
        df_participantes['RENDA_FAMILIAR_SM'] = df_participantes['RENDA_FAMILIAR_SM'].map(cor_renda_map_sm)

    media_q006_por_uf = df_participantes.groupby('SG_UF_PROVA')['RENDA_FAMILIAR_SM'].transform('mean')
    df_participantes['RENDA_FAMILIAR_SM'] = df_participantes['RENDA_FAMILIAR_SM'].fillna(media_q006_por_uf)

    df_participantes['RENDA_FAMILIAR_SM'] = df_participantes['RENDA_FAMILIAR_SM'].fillna(df_participantes['RENDA_FAMILIAR_SM'].mean())

In [ ]:
df_participantes.isnull().sum()*100

In [ ]:
df_participantes.info()

In [ ]:
df_participantes = df_participantes[df_participantes['IN_TREINEIRO'] != 1]
df_participantes = df_participantes.drop(columns=['IN_TREINEIRO'])

In [ ]:
df_participantes['MUNICIPIO'] = df_participantes['MUNICIPIO'].str.upper()

In [ ]:
df_participantes.info()

In [ ]:
analise_cols = ['RENDA_FAMILIAR_SM']

for col in analise_cols:
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4))
    
    df_participantes[col].hist(ax=ax1, density=True)
    ax1.set_title(f'Histograma - {col}')
    ax1.set_xlabel(col)
    
    df_participantes.boxplot(column=col, ax=ax2)
    ax2.set_title(f'Box Plot - {col}')
    
    plt.tight_layout()
    plt.show()

In [ ]:
df_participantes['RENDA_FAMILIAR_SM_LOG'] = np.log1p(df_participantes['RENDA_FAMILIAR_SM'])

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4))

df_participantes['RENDA_FAMILIAR_SM_LOG'].hist(ax=ax1, bins=30)
ax1.set_title('Histograma - RENDA_FAMILIAR_SM_LOG')
ax1.set_xlabel('RENDA_FAMILIAR_SM_LOG')

df_participantes.boxplot(column='RENDA_FAMILIAR_SM_LOG', ax=ax2)
ax2.set_title('Box Plot - RENDA_FAMILIAR_SM_LOG')

plt.tight_layout()
plt.show()


In [ ]:
df_tratamento_outlier = df_participantes.copy()

valores_q1 = [0.25, 0.2, 0.15, 0.1, 0.05]
valores_q3 = [0.75, 0.8, 0.85, 0.9, 0.95]

col = 'RENDA_FAMILIAR_SM_LOG'

for q1_val, q3_val in zip(valores_q1, valores_q3):
    df_col_filtrado = df_tratamento_outlier.copy()

    Q1 = df_tratamento_outlier[col].quantile(q1_val)
    Q3 = df_tratamento_outlier[col].quantile(q3_val)
    IQR = Q3 - Q1

    df_col_filtrado = df_col_filtrado[
        (df_col_filtrado[col] >= Q1 - 1.5 * IQR) &
        (df_col_filtrado[col] <= Q3 + 1.5 * IQR)
    ]

    proporcao_removida = (
        (df_tratamento_outlier.shape[0] - df_col_filtrado.shape[0])
        / df_tratamento_outlier.shape[0]
    ) * 100

    if proporcao_removida <= 5:
        print(f'Tratamento de outliers de renda | Q1={q1_val}, Q3={q3_val} {proporcao_removida:.2f}% removidos')
        df_tratamento_outlier = df_col_filtrado.copy()
        break
else:
    print(f'Nenhum corte adequado para {col} — mantendo dados originais\n')

df_participantes = df_tratamento_outlier.copy()
df_participantes = df_participantes.drop(columns=['RENDA_FAMILIAR_SM_LOG'])


In [ ]:
df_municipio = df_participantes.groupby('MUNICIPIO').agg(
    RENDA_FAMILIAR_SM_MEDIA=('RENDA_FAMILIAR_SM', 'mean')
).reset_index()

In [ ]:
df_municipio.head()

## Tratamento Resultados

In [ ]:
df_resultado.head()

In [ ]:
df_resultado.isnull().mean()*100

In [ ]:
df_resultado = df_resultado[df_resultado['TP_PRESENCA_CN'] == 1]
df_resultado = df_resultado[df_resultado['TP_PRESENCA_CH'] == 1]
df_resultado = df_resultado[df_resultado['TP_PRESENCA_LC'] == 1]
df_resultado = df_resultado[df_resultado['TP_PRESENCA_MT'] == 1]

df_resultado = df_resultado.drop(columns=['TP_PRESENCA_CN', 'TP_PRESENCA_CH', 'TP_PRESENCA_LC', 'TP_PRESENCA_MT'])

In [ ]:
df_resultado.isnull().mean()*100

In [ ]:
df_resultado['MUNICIPIO'] = df_resultado['MUNICIPIO'].str.upper()

In [ ]:
df_resultado.head()

In [ ]:
analise_notas = ['NU_NOTA_CN', 'NU_NOTA_CH', 'NU_NOTA_LC', 'NU_NOTA_MT', 'NU_NOTA_REDACAO']

In [ ]:
for col in analise_notas:
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4))

    if col == 'NU_NOTA_REDACAO':
        ax1.hist(df_resultado[col].dropna(), bins=50, density=True, color='blue')
    else:
        ax1.hist(df_resultado[col].dropna(), bins=150, density=True, color='blue')
    ax1.set_title(f'Histograma - {col}')
    ax1.set_xlabel(col)

    ax2.boxplot(df_resultado[col].dropna(), vert=True)
    ax2.set_title(f'Box Plot - {col}')
    ax2.set_ylabel(col)

    plt.tight_layout()
    plt.show()


In [ ]:
df_tratamento_outlier = df_resultado.copy()

valores_q1 = [0.25, 0.2, 0.15, 0.1, 0.05]
valores_q3 = [0.75, 0.8, 0.85, 0.9, 0.95]

n_original = df_resultado.shape[0]

for q1_val, q3_val in zip(valores_q1, valores_q3):
    
    df_temp = df_resultado.copy()
    
    for col in analise_notas:
        Q1 = df_resultado[col].quantile(q1_val)
        Q3 = df_resultado[col].quantile(q3_val)
        IQR = Q3 - Q1
        
        df_temp = df_temp[
            (df_temp[col] >= Q1 - 1.5 * IQR) & 
            (df_temp[col] <= Q3 + 1.5 * IQR)
        ]
    
    proporcao_total = (
        (n_original - df_temp.shape[0]) / n_original
    ) * 100
    

    if proporcao_total <= 5:
        print(f'Tratamento de outliers de notas | Q1={q1_val}, Q3={q3_val} {proporcao_total:.2f}% removidos')
        df_tratamento_outlier = df_temp.copy()
        break
else:
    print('Nenhum corte global válido — mantendo dados')

print(f'\nTotal removido final: {((n_original - df_tratamento_outlier.shape[0]) / n_original) * 100:.2f}%')

df_resultado = df_tratamento_outlier.copy()

In [ ]:
for col in analise_notas:
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4))

    if col == 'NU_NOTA_REDACAO':
        ax1.hist(df_resultado[col].dropna(), bins=50, density=True, color='red')
    else:
        ax1.hist(df_resultado[col].dropna(), bins=150, density=True, color='red')
    
    ax1.set_title(f'Histograma - {col}')
    ax1.set_xlabel(col)

    ax2.boxplot(df_resultado[col].dropna(), vert=True)
    ax2.set_title(f'Box Plot - {col}')
    ax2.set_ylabel(col)

    plt.tight_layout()
    plt.show()

In [ ]:
df_resultado.head()

In [ ]:
df_resultado = df_resultado.groupby('COD_MUNICIPIO').agg(
    UF=('SG_UF_PROVA', 'first'),
    MUNICIPIO=('MUNICIPIO', 'first'),
    QTD_PARTICIPANTES=('COD_MUNICIPIO', 'size'),
    NOTA_CN_MEDIA=('NU_NOTA_CN', 'mean'),
    NOTA_CH_MEDIA=('NU_NOTA_CH', 'mean'),
    NOTA_LC_MEDIA=('NU_NOTA_LC', 'mean'),
    NOTA_MT_MEDIA=('NU_NOTA_MT', 'mean'),
    NOTA_REDACAO_MEDIA=('NU_NOTA_REDACAO', 'mean')
).reset_index()

df_resultado['MEDIA_GERAL'] = df_resultado[['NOTA_CN_MEDIA', 'NOTA_CH_MEDIA', 'NOTA_LC_MEDIA', 'NOTA_MT_MEDIA', 'NOTA_REDACAO_MEDIA']].mean(axis=1)

In [ ]:
df_municipio_clustering = df_resultado.merge(
    df_municipio,
    on='COD_MUNICIPIO',
    how='left'
)

X_scaled_municipio = df_municipio_clustering.copy()

X_scaled_municipio = X_scaled_municipio.drop(columns=['MUNICIPIO', 'RENDA_FAMILIAR_SM_MEDIA', 'UF', 'QTD_PARTICIPANTES', 'MEDIA_GERAL'])

col_scatter = ['NOTA_CN_MEDIA', 'NOTA_CH_MEDIA', 'NOTA_LC_MEDIA', 'NOTA_MT_MEDIA', 'NOTA_REDACAO_MEDIA']
scaler = StandardScaler()
X_scaled_municipio[col_scatter] = scaler.fit_transform(X_scaled_municipio[col_scatter])


In [ ]:
colunas_notas_media = ['NOTA_CN_MEDIA', 'NOTA_CH_MEDIA', 'NOTA_LC_MEDIA', 'NOTA_MT_MEDIA', 'NOTA_REDACAO_MEDIA', 'MEDIA_GERAL']

In [ ]:
def media_ponderada(grupo, coluna, peso="QTD_PARTICIPANTES"):
    d = grupo[[coluna, peso]].dropna()
    soma_pesos = d[peso].sum()
    return (d[coluna] * d[peso]).sum() / soma_pesos if soma_pesos != 0 else pd.NA

df_estado_clustering = (
    df_municipio_clustering.groupby("UF")
    .apply(
        lambda g: pd.Series(
            {
                "QTD_PARTICIPANTES": g["QTD_PARTICIPANTES"].sum(),
                "RENDA_FAMILIAR_SM_MEDIA": g["RENDA_FAMILIAR_SM_MEDIA"].mean(),
                **{col: media_ponderada(g, col) for col in colunas_notas_media},
            }
        )
    )
    .reset_index()
    .sort_values("UF")
    .reset_index(drop=True)
)


X_scaled_estado = df_estado_clustering.copy()
X_scaled_estado = X_scaled_estado.drop(columns=['RENDA_FAMILIAR_SM_MEDIA', 'UF', 'QTD_PARTICIPANTES', 'MEDIA_GERAL'])
col_scatter = ['NOTA_CN_MEDIA', 'NOTA_CH_MEDIA', 'NOTA_LC_MEDIA', 'NOTA_MT_MEDIA', 'NOTA_REDACAO_MEDIA']
scaler = StandardScaler()
X_scaled_estado[col_scatter] = scaler.fit_transform(X_scaled_estado[col_scatter])

In [ ]:
df_municipio_clustering = df_municipio_clustering.sort_values('MEDIA_GERAL', ascending=False).reset_index(drop=True)
df_municipio_clustering.head()

In [ ]:
X_scaled_municipio.head()

In [ ]:
df_estado_clustering['QTD_PARTICIPANTES'] = df_estado_clustering['QTD_PARTICIPANTES'].astype('int64')
df_estado_clustering = df_estado_clustering.sort_values('MEDIA_GERAL', ascending=False).reset_index(drop=True)  

df_estado_clustering.head()

In [ ]:
X_scaled_estado.head()